# Lesson 07 - 計画デザインパターン

このノートブックでは、Microsoft Agent Framework を使用して AI エージェントの**計画デザインパターン**を示します。
複雑な旅行リクエストを構造化されたサブタスクに分解し、それらを専門のエージェントに割り当て、
構造化された出力を Pydantic モデルで活用して計画を実行する方法を学びます。


## セットアップ


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## タスク分解

タスク分解はプランニングデザインパターンの核心です。単一のエージェントに複雑なリクエスト全体を処理させるのではなく、問題をより小さく明確に定義された**サブタスク**に分割します。各サブタスクは専門エージェント（例：フライト、ホテル、アクティビティ）に割り当てられ、明確な優先順位と依存関係の順序が設定されます。

このアプローチは以下の利点を提供します:
- **明確さ**: 各サブタスクは単一の責任を持つ。
- **並行性**: 独立したサブタスクは同時に実行可能。
- **信頼性**: 失敗は個々のサブタスクに限定される。
- **予算追跡**: コストはサブタスクごとに見積もられ、合算される。


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## 構造化出力を用いたプランニングエージェントの作成

プランニングエージェントは**フロントデスクコーディネーター**として機能します。高レベルな旅行リクエストが与えられると、リクエストをサブタスクに分解し、優先順位を設定し、依存関係を特定する構造化された`TravelPlan`を生成します。これにより、コンシェルジュや実行層が作業を遂行できるようにします。


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## 専門ツールを使ったプランの実行

フロントデスクの担当者が構造化されたプランを作成したら、**コンシェルジュエージェント**がそれを実行します。
各専門ツールはサブタスクの1つのカテゴリ（フライト、ホテル、アクティビティ）を担当します。コンシェルジュはプランのサブタスクを依存関係の順に繰り返し、それぞれを適切なツールに割り当てます。


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Summary

このレッスンでは、AIエージェントのための**プランニングデザインパターン**について学びました：

1. **タスク分解** — フロントデスクのプランニングエージェントは、複雑な旅行リクエストをPydanticモデルを使って構造化されたサブタスクに分解し、それぞれに優先順位と依存関係を付けて専門エージェントに割り当てます。
2. **構造化された出力** — `response_format`を渡すことで、エージェントは自由形式のテキストの代わりに検証された`TravelPlan`オブジェクトを返し、下流の処理を信頼性の高いものにします。
3. **プランの実行** — コンシェルジュエージェントは、専門ツール（`book_flight`、`reserve_hotel`、`book_activity`）を使ってサブタスクを順に実行し、結果を報告します。

このパターンは、*何をするか*（プランニング）を*どうやってするか*（実行）から分離し、エージェントをよりモジュール式でテスト可能、かつ拡張しやすくします。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：  
本書類はAI翻訳サービス[Co-op Translator](https://github.com/Azure/co-op-translator)を用いて翻訳されています。正確性の確保に努めておりますが、自動翻訳には誤りや不正確な部分が含まれる可能性があります。原文の言語によるオリジナル文書を正式な情報源としてご参照ください。重要な情報に関しては、専門の人間による翻訳を推奨いたします。本翻訳の利用に起因するいかなる誤解や誤訳についても、当方は一切責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
